# TP – Decision Transformer para recomendación (Netflix sintético)

Notebook único. **Ejecutarlo desde `2026/tp`** (donde está la carpeta `protocolo/`).

Orden:
1. Configuración e imports
2. Modelo (`DecisionTransformer`, loss, máscara, test de no fuga, `DTPolicy`)
3. Funciones de entrenamiento
4. Tests del modelo
5. Datos
6. Entrenamiento
7. Curvas de pérdida
8. Evaluación: barrido de retornos pedidos en validación
9. Evaluación final en test (una sola vez)

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from protocolo.datos import Episodes, load_test, load_train, make_episodes, num_items, split_by_user
from protocolo.evaluar import baseline_factories, evaluate
from protocolo.simulador import SessionConfig

DATASET = "netflix"
SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT = Path(f"outputs/dt/{DATASET}/seed{SEED}")
print("device:", DEVICE)

device: cpu


## 2. Modelo

In [2]:
class DecisionTransformer(nn.Module):
    """Tokens por paso: (R̂_t, s_t, a_t) intercalados. a_t se predice desde la salida de s_t."""

    def __init__(self, n_items, horizon=20, d_model=128, n_layers=3, n_heads=4, n_groups=8, dropout=0.1):
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model debe ser divisible por n_heads")
        self.n_items = n_items
        self.horizon = horizon
        self.n_groups = n_groups

        self.rtg_emb = nn.Linear(1, d_model)
        self.group_emb = nn.Embedding(n_groups + 1, d_model)  # +1: grupo desconocido
        self.prev_rating_emb = nn.Linear(1, d_model)
        self.item_emb = nn.Embedding(n_items, d_model)
        self.step_emb = nn.Embedding(horizon, d_model)
        self.type_emb = nn.Embedding(3, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=4 * d_model, dropout=dropout,
            activation="gelu", norm_first=True, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False)
        self.ln_out = nn.LayerNorm(d_model)

        for emb in (self.item_emb, self.group_emb, self.step_emb, self.type_emb):
            nn.init.normal_(emb.weight, std=0.02)

    def forward(self, rtg, items, state, groups):
        """Logits (B, T, n_items) para a_t, calculados desde el token s_t."""
        if items.dim() != 2:
            raise ValueError("items debe tener forma (B, T)")
        B, T = items.shape
        if T > self.horizon:
            raise ValueError(f"T={T} supera el horizonte {self.horizon}")

        dtype = self.item_emb.weight.dtype
        positions = torch.arange(T, device=items.device)
        type_ids = torch.arange(3, device=items.device)

        rtg_tokens = self.rtg_emb(rtg.to(dtype).unsqueeze(-1) / 100.0)
        state_tokens = self.group_emb(groups).unsqueeze(1) + self.prev_rating_emb(
            state.to(dtype).unsqueeze(-1) / 5.0
        )
        item_tokens = self.item_emb(items)

        step = self.step_emb(positions).unsqueeze(0)
        tokens = torch.stack(
            [
                rtg_tokens + step + self.type_emb(type_ids[0]),
                state_tokens + step + self.type_emb(type_ids[1]),
                item_tokens + step + self.type_emb(type_ids[2]),
            ],
            dim=2,
        ).reshape(B, 3 * T, -1)

        causal = torch.triu(torch.ones(3 * T, 3 * T, dtype=torch.bool, device=items.device), diagonal=1)
        hidden = self.ln_out(self.encoder(tokens, mask=causal))
        state_out = hidden[:, 1::3]
        return state_out @ self.item_emb.weight.T


def seen_before_mask(items, n_items):
    """(B, T, n_items): True si el ítem ya se mostró antes del paso t en el episodio."""
    one_hot = F.one_hot(items, num_classes=n_items)
    seen = one_hot.cumsum(dim=1) > 0
    return torch.cat([torch.zeros_like(seen[:, :1]), seen[:, :-1]], dim=1)


def dt_loss(model, batch):
    logits = model(batch["rtg"], batch["items"], batch["state"], batch["groups"])
    logits = logits.masked_fill(seen_before_mask(batch["items"], logits.shape[-1]), float("-inf"))
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), batch["items"].reshape(-1))


def check_no_leak(model, batch, t=7, atol=1e-5):
    """La salida del paso t no debe depender de a_t ni del futuro."""
    was_training = model.training
    model.eval()
    with torch.no_grad():
        base = model(batch["rtg"], batch["items"], batch["state"], batch["groups"])[:, t]
        items2 = batch["items"].clone()
        items2[:, t:] = torch.randint(0, model.n_items, items2[:, t:].shape, device=items2.device)
        rtg2 = batch["rtg"].clone();     rtg2[:, t + 1:] = 0.0
        state2 = batch["state"].clone(); state2[:, t + 1:] = 0.0
        out = model(rtg2, items2, state2, batch["groups"])[:, t]
    if was_training:
        model.train()
    return bool(torch.allclose(base, out, atol=atol))


class DTPolicy:
    """Política autorregresiva compatible con simulador.Policy (reset / scores)."""

    def __init__(self, model, target_return):
        self.model = model.eval()
        self.target = float(target_return)
        self.group = model.n_groups
        self.horizon = model.horizon

    def reset(self, group, horizon):
        self.group = self.model.n_groups if group is None else int(group)
        self.horizon = int(horizon)

    @torch.no_grad()
    def scores(self, items_seen, ratings_seen):
        items_seen = np.asarray(items_seen, dtype=np.int64)
        ratings_seen = np.asarray(ratings_seen, dtype=np.float64)
        t = len(items_seen)

        rtg = np.empty(t + 1, dtype=np.float32)
        rtg[0] = self.target
        if t:
            rtg[1:] = self.target - np.cumsum(ratings_seen)

        items = np.zeros(t + 1, dtype=np.int64)
        items[:t] = items_seen
        prev = np.zeros(t + 1, dtype=np.float32)
        prev[1:] = ratings_seen

        device = next(self.model.parameters()).device
        batch = {
            "rtg": torch.as_tensor(rtg, device=device).unsqueeze(0),
            "items": torch.as_tensor(items, device=device).unsqueeze(0),
            "state": torch.as_tensor(prev, device=device).unsqueeze(0),
            "groups": torch.tensor([self.group], device=device),
        }
        logits = self.model(**batch)[0, t].clone()
        logits[items_seen] = float("-inf")
        return logits.cpu().numpy()

## 3. Funciones de entrenamiento

In [3]:
def episodes_to_tensors(ep, device="cpu"):
    ratings = torch.as_tensor(ep.ratings, dtype=torch.float32, device=device)
    items = torch.as_tensor(ep.items, dtype=torch.long, device=device)
    groups = torch.as_tensor(ep.groups, dtype=torch.long, device=device)
    state = torch.zeros_like(ratings)
    state[:, 1:] = ratings[:, :-1]          # estado = rating del paso anterior
    rtg = torch.as_tensor(np.ascontiguousarray(ep.returns_to_go()), dtype=torch.float32, device=device)
    return {"rtg": rtg, "items": items, "state": state, "groups": groups, "ratings": ratings}


@torch.no_grad()
def mean_loss(model, data, batch_size=512):
    was_training = model.training
    model.eval()
    n = data["items"].shape[0]
    total = 0.0
    for start in range(0, n, batch_size):
        batch = {k: v[start:start + batch_size] for k, v in data.items() if k != "ratings"}
        total += dt_loss(model, batch).item() * batch["items"].numel()
    if was_training:
        model.train()
    return total / (n * data["items"].shape[1])


def train_dt(train_ep, val_ep, n_items, seed=0, epochs=10, batch_size=256, lr=1e-3, weight_decay=0.01,
             group_dropout=0.3, grad_clip=1.0, d_model=128, n_layers=3, n_heads=4, dropout=0.1,
             max_batches=None, device="cpu", output=None, verbose=True):
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    horizon = int(train_ep.items.shape[1])
    train_data = episodes_to_tensors(train_ep, device)
    val_data = episodes_to_tensors(val_ep, device)

    model = DecisionTransformer(n_items=n_items, horizon=horizon, d_model=d_model, n_layers=n_layers,
                                n_heads=n_heads, dropout=dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_episodes = train_data["items"].shape[0]
    steps_per_epoch = max(1, (n_episodes + batch_size - 1) // batch_size)
    if max_batches is not None:
        steps_per_epoch = min(steps_per_epoch, max_batches)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=max(1, epochs * steps_per_epoch))

    config = dict(n_items=n_items, horizon=horizon, d_model=d_model, n_layers=n_layers, n_heads=n_heads,
                  dropout=dropout, seed=seed, epochs=epochs, batch_size=batch_size, lr=lr,
                  weight_decay=weight_decay, group_dropout=group_dropout)

    history, best_val, best_state = [], float("inf"), None

    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(n_episodes, generator=generator).to(device)
        running, batches = 0.0, 0

        for step in range(steps_per_epoch):
            index = order[step * batch_size:(step + 1) * batch_size]
            if index.numel() == 0:
                continue
            batch = {k: v[index] for k, v in train_data.items() if k != "ratings"}
            if group_dropout > 0:   # a veces ocultamos el grupo para aprender cold-start
                drop = torch.rand(batch["groups"].shape, generator=generator).to(device) < group_dropout
                batch["groups"] = torch.where(drop, torch.full_like(batch["groups"], model.n_groups), batch["groups"])
            loss = dt_loss(model, batch)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            scheduler.step()
            running += loss.item()
            batches += 1

        train_loss = running / max(1, batches)
        val_loss = mean_loss(model, val_data)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if verbose:
            print(f"epoch {epoch:3d}/{epochs} | train {train_loss:.4f} | val {val_loss:.4f} | "
                  f"mejor {best_val:.4f} | lr {scheduler.get_last_lr()[0]:.2e}")

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

    if output is not None:
        out = Path(output)
        out.mkdir(parents=True, exist_ok=True)
        torch.save({"state_dict": model.state_dict(), "config": config, "best_val_loss": best_val}, out / "modelo.pt")
        pd.DataFrame(history).to_csv(out / "historial.csv", index=False)
        (out / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
        if verbose:
            print(f"checkpoint e historial en {out}")

    return model, pd.DataFrame(history), best_val


def load_dt(path, device="cpu"):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    c = ckpt["config"]
    model = DecisionTransformer(n_items=c["n_items"], horizon=c["horizon"], d_model=c["d_model"],
                                n_layers=c["n_layers"], n_heads=c["n_heads"], dropout=c["dropout"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    return model.eval()

## 4. Tests del modelo
Son los mismos de `test_dt.py`, corren acá mismo con `assert`. Si alguno falla, no sigas al entrenamiento.

In [4]:
def make_batch(B=4, T=20, n_items=50, n_groups=8, seed=0):
    g = torch.Generator().manual_seed(seed)
    items = torch.stack([torch.randperm(n_items, generator=g)[:T] for _ in range(B)])
    ratings = torch.randint(1, 6, (B, T), generator=g).float()
    state = torch.zeros_like(ratings); state[:, 1:] = ratings[:, :-1]
    rtg = torch.flip(torch.cumsum(torch.flip(ratings, dims=[1]), dim=1), dims=[1])
    groups = torch.randint(0, n_groups, (B,), generator=g)
    return {"items": items, "state": state, "rtg": rtg, "groups": groups}


def make_small_model(n_items=50):
    torch.manual_seed(0)
    return DecisionTransformer(n_items=n_items, horizon=20, d_model=32, n_layers=1, n_heads=2, dropout=0.0)


def run_tests():
    # máscara de ítems ya vistos
    mask = seen_before_mask(torch.tensor([[17, 302, 45, 17]]), n_items=400)
    assert mask.shape == (1, 4, 400) and not mask[0, 0].any()
    assert mask[0, 1, 17] and mask[0, 2, 17] and mask[0, 2, 302] and not mask[0, 2, 45] and mask[0, 3, 17]

    model, batch = make_small_model(), make_batch()
    out = model(batch["rtg"], batch["items"], batch["state"], batch["groups"])
    assert out.shape == (4, 20, 50), "forma de salida"
    assert torch.isfinite(dt_loss(model, batch)), "loss finita"
    assert check_no_leak(model, batch, t=7), "FUGA: la salida en t depende de a_t o del futuro"

    # la política enmascara lo ya visto
    pol = DTPolicy(make_small_model(n_items=40), target_return=80.0)
    pol.reset(group=None, horizon=20)
    s = pol.scores(np.array([], dtype=np.int64), np.array([], dtype=np.float64))
    assert s.shape == (40,) and np.isfinite(s).all()
    seen = np.array([1, 2, 3])
    pol.reset(group=3, horizon=20)
    s = pol.scores(seen, np.array([4.0, 5.0, 3.0]))
    assert np.isneginf(s[seen]).all() and np.isfinite(s[~np.isin(np.arange(40), seen)]).all()

    # el modelo realmente usa rtg y state actuales
    with torch.no_grad():
        base = model(batch["rtg"], batch["items"], batch["state"], batch["groups"])[:, 5]
        r2 = batch["rtg"].clone(); r2[:, 5] += 25.0
        assert not torch.allclose(base, model(r2, batch["items"], batch["state"], batch["groups"])[:, 5])
        s2 = batch["state"].clone(); s2[:, 5] += 2.5
        assert not torch.allclose(base, model(batch["rtg"], batch["items"], s2, batch["groups"])[:, 5])
    print("OK: todos los tests pasaron")

run_tests()

OK: todos los tests pasaron


## 5. Datos
`split_by_user`: 90 % ajuste / 10 % validación (estratificado, semilla 0). Episodios de 20 pasos.

In [5]:
train_all = load_train(DATASET)
train, val = split_by_user(train_all)
train_ep = make_episodes(train, length=20)
val_ep = make_episodes(val, length=20)
N_ITEMS = num_items(DATASET)

print("episodios train:", train_ep.items.shape, "| val:", val_ep.items.shape, "| n_items:", N_ITEMS)
print("retorno medio de un episodio de train:", float(train_ep.ratings.sum(1).mean()), "(el protocolo dice ≈ 64,3)")

episodios train: (73827, 20) | val: (8275, 20) | n_items: 752
retorno medio de un episodio de train: 64.29808877510938 (el protocolo dice ≈ 64,3)


## 6. Entrenamiento
Primero una corrida corta para verificar que todo anda; después la real.

In [6]:
# Prueba rápida (1 epoch, 5 batches)
_ = train_dt(train_ep, val_ep, N_ITEMS, seed=SEED, epochs=1, max_batches=5, device=DEVICE, output=None)

epoch   1/1 | train 6.6299 | val 6.6270 | mejor 6.6270 | lr 1.88e-04


In [ ]:
model, history, best_val = train_dt(
    train_ep, val_ep, N_ITEMS, seed=SEED, epochs=10, batch_size=256, lr=1e-3,
    d_model=128, n_layers=3, n_heads=4, dropout=0.1, group_dropout=0.3,
    device=DEVICE, output=OUTPUT,
)
print(f"mejor val CE: {best_val:.4f} | referencia uniforme log(n_items) ≈ {np.log(N_ITEMS):.2f}")

epoch   1/10 | train 6.6154 | val 6.6126 | mejor 6.6126 | lr 2.81e-04
epoch   2/10 | train 6.6139 | val 6.6125 | mejor 6.6125 | lr 7.61e-04
epoch   3/10 | train 6.6084 | val 6.6082 | mejor 6.6082 | lr 1.00e-03


## 7. Curvas de pérdida
Recordá: la loss baja no implica buena política. Lo que cuenta es la evaluación del protocolo.

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(history["epoch"], history["train_loss"], label="train")
plt.plot(history["epoch"], history["val_loss"], label="val")
plt.axhline(np.log(N_ITEMS), ls="--", c="gray", label="uniforme")
plt.xlabel("epoch"); plt.ylabel("cross-entropy"); plt.legend(); plt.tight_layout(); plt.show()

## 8. Evaluación en validación: barrido de retorno pedido

Se usa `evaluate()` del protocolo con `DTPolicy(model, target_return=R)` sobre **validación** (catálogo restringido a los ítems que cada usuario valoró; lo maneja `Session`).

Para que no tarde horas, el barrido usa un subconjunto de usuarios de validación y 1 semilla (`N_USERS_VAL`, `SEEDS_VAL`). Subilos si tenés tiempo/GPU.

También se evalúan `item_mean` y `group_mean` como referencia: **el DT tiene que superar a `item_mean`**.

In [ ]:
RETORNOS = [50, 60, 64, 70, 80, 90, 100, 110, 125]   # el retorno medio de train es ≈ 64,3
N_USERS_VAL = 400
SEEDS_VAL = (0,)
N_SEEDS_TEST = (0, 1, 2)

def correr(policy, users, group_visible=True, warmup=0, seeds=(0,)):
    """Corre el protocolo y devuelve métricas promedio (primero por usuario, como metricas.summarize)."""
    config = SessionConfig(horizon=20, warmup=warmup, group_visible=group_visible)
    df = evaluate(policy, users, N_ITEMS, config, seeds)
    return df.groupby("user_id")[["normalized", "return", "hit_rate"]].mean().mean()

rng = np.random.default_rng(0)
idx = np.sort(rng.choice(len(val), size=min(N_USERS_VAL, len(val)), replace=False))
val_sub = [val[i] for i in idx]

# Referencias en validación
refs = baseline_factories(train, N_ITEMS)
filas = []
for visible in (True, False):
    for nombre in ("item_mean", "group_mean"):
        m = correr(refs[nombre](), val_sub, group_visible=visible, seeds=SEEDS_VAL)
        filas.append({"policy": nombre, "group_visible": visible, "R_pedido": np.nan, **m.to_dict()})

# Barrido de R
for visible in (True, False):
    for R in RETORNOS:
        m = correr(DTPolicy(model, target_return=R), val_sub, group_visible=visible, seeds=SEEDS_VAL)
        filas.append({"policy": "DT", "group_visible": visible, "R_pedido": R, **m.to_dict()})
        print(f"visible={visible!s:5} R={R:4d} -> normalized {m['normalized']:.3f} | retorno obtenido {m['return']:.1f}")

barrido = pd.DataFrame(filas)
barrido

In [ ]:
dt_res = barrido[barrido.policy == "DT"]
ret_ep = train_ep.ratings.sum(1)
lo, hi = np.percentile(ret_ep, [5, 95])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for visible, g in dt_res.groupby("group_visible"):
    etiqueta = "grupo visible" if visible else "grupo oculto"
    ax[0].plot(g.R_pedido, g["return"], marker="o", label=etiqueta)
    ax[1].plot(g.R_pedido, g.normalized, marker="o", label=etiqueta)
ax[0].plot(RETORNOS, RETORNOS, "--", c="gray", label="obtenido = pedido")
for a in ax:
    a.axvspan(lo, hi, color="gray", alpha=0.15, label="soporte train (p5–p95)")
    a.set_xlabel("retorno pedido R")
ax[0].set_ylabel("retorno obtenido"); ax[0].legend(fontsize=8)
for visible in (True, False):
    ref = barrido[(barrido.policy == "item_mean") & (barrido.group_visible == visible)]["normalized"].iloc[0]
    ax[1].axhline(ref, ls=":", c="C3" if visible else "C4", label=f"item_mean ({'vis.' if visible else 'oculto'})")
ax[1].set_ylabel("normalized"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 9. Evaluación final en test (**una sola vez**)

`R` se elige en validación (el que maximiza `normalized` promediando grupo visible y oculto). Se evalúa con 3 semillas, warm-up 0 y 5, grupo visible y oculto, junto con las líneas base.

Ojo con el tiempo: son 1.600 usuarios × 20 pasos × 3 semillas × 4 condiciones. Si querés probar antes, poné `N_USERS_TEST = 200`, pero la tabla final tiene que ser con todos.

In [ ]:
N_USERS_TEST = None   # None = todos los usuarios de test

mejor = dt_res.groupby("R_pedido")["normalized"].mean()
R_ELEGIDO = float(mejor.idxmax())
print("R elegido en validación:", R_ELEGIDO)

test_users = load_test(DATASET)
if N_USERS_TEST:
    test_users = test_users[:N_USERS_TEST]

politicas = {"DT": lambda: DTPolicy(model, target_return=R_ELEGIDO),
             **{k: v for k, v in baseline_factories(train, N_ITEMS).items()
                if k in ("random", "item_mean", "group_mean", "group_posterior")}}

filas = []
for nombre, make in politicas.items():
    for warmup in (0, 5):
        for visible in (True, False):
            m = correr(make(), test_users, group_visible=visible, warmup=warmup, seeds=N_SEEDS_TEST)
            filas.append({"policy": nombre, "warmup": warmup, "group_visible": visible, **m.to_dict()})
tabla_test = pd.DataFrame(filas)
tabla_test.pivot_table(index="policy", columns=["warmup", "group_visible"], values="normalized").round(3)